# Exploratory Data Analysis

In [1]:
import pandas as pd

In [38]:
movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data/ratings.csv")
tags = pd.read_csv("../data/tags.csv")

## overveiw of all files:

In [46]:
display(movies.shape, ratings.shape, tags.shape)
display(movies.count(), ratings.count(), tags.count())

(86537, 3)

(33832162, 4)

(2328315, 4)

movieId    86537
title      86537
genres     86537
dtype: int64

userId       33832162
movieId      33832162
rating       33832162
timestamp    33832162
dtype: int64

userId       2328315
movieId      2328315
tag          2328298
timestamp    2328315
dtype: int64

In [36]:
movies_info = pd.DataFrame({
    "column": movies.columns,
    "dtype": movies.dtypes.astype(str).values,
    "missing_values": movies.isna().sum().values,
    "n_unique": movies.nunique(dropna=False).values,
    "duplicates": movies.duplicated().sum()
})

ratings_info = pd.DataFrame({
    "column": ratings.columns,
    "dtype": ratings.dtypes.astype(str).values,
    "missing_values": ratings.isna().sum().values,
    "n_unique": ratings.nunique(dropna=False).values,
    "duplicates": ratings.duplicated().sum()
})

tags_info = pd.DataFrame({
    "column": tags.columns,
    "dtype": tags.dtypes.astype(str).values,
    "missing_values": tags.isna().sum().values,
    "n_unique": tags.nunique(dropna=False).values,
    "duplicates": tags.duplicated().sum()
})

display(movies_info)
display(ratings_info)
display(tags_info)


,column,dtype,missing_values,n_unique,duplicates
0,movieId,int64,0,86537,0
1,title,str,0,86330,0
2,genres,str,0,1796,0


,column,dtype,missing_values,n_unique,duplicates
0,userId,int64,0,330975,0
1,movieId,int64,0,83239,0
2,rating,float64,0,10,0
3,timestamp,int64,0,27419646,0


,column,dtype,missing_values,n_unique,duplicates
0,userId,int64,0,25280,0
1,movieId,int64,0,53452,0
2,tag,str,17,153950,0
3,timestamp,int64,0,1607772,0


## Ratings

### How many ratings per userId are there? 
-  Are there features with few observations? (might be important later to filter out users and movies with few observations)

In [4]:
ratings_per_user = ratings.groupby("userId").size().reset_index(name="n_ratings")
ratings_per_movie = ratings.groupby("movieId").size().reset_index(name="n_ratings")

display(ratings_per_user["n_ratings"].describe())
display(ratings_per_movie["n_ratings"].describe())


count    330975.00000
mean        102.21969
std         232.15453
min           1.00000
25%          15.00000
50%          31.00000
75%          98.00000
max       33332.00000
Name: n_ratings, dtype: float64

count     83239.000000
mean        406.446041
std        2806.975876
min           1.000000
25%           2.000000
50%           5.000000
75%          26.000000
max      122296.000000
Name: n_ratings, dtype: float64

**Ratings per user**

This means:

- there are 330,975 users
- the median user has given only 31 ratings
- 25% of users have given at most 15 ratings
- 75% of users have given at most 98 ratings
- the mean is 102, which is much higher than the median of 31
- the maximum value, 33,332, is extremely high

*Conclusion*

The distribution is strongly right-skewed:

- most users are only mildly active
- a smaller number of users are very active
- the very active users pull the mean upward substantially

This means that if you build a collaborative filtering system, you should probably filter out users with very few ratings, since their profiles will be weak.

A reasonable first threshold could be, for example:

- at least 20 ratings
- or at least 30 ratings


**Ratings per movie**

This means:

- there are 83,239 movies with ratings
- the median movie has only 5 ratings
- 25% of movies have at most 2 ratings
- 75% of movies have at most 26 ratings
- the mean is 406, which is far higher than the median of 5
- the maximum value, 122,296, shows that a few movies are extremely popular

*Conclusion*

Here too, the distribution is extremely right-skewed:

- most movies have very few ratings
- a small number of movies have a very large number of ratings
- popular blockbuster movies dominate the overall totals

This means that you will likely want to filter out movies with very few ratings if they are to be used in recommendations or in user profiles.

A reasonable first threshold could be:

- at least 5 ratings
- at least 10 ratings
- or more, depending on how strict you want to be


Checking **duplicates** in ratings

In [5]:
exact_duplicates = ratings.duplicated().sum()
print("Number of exact duplicate rows:", exact_duplicates)

Number of exact duplicate rows: 0


In [6]:
user_movie_duplicates = (
    ratings.groupby(["userId", "movieId"])
    .size()
    .reset_index(name="count")
)

repeated_user_movie = user_movie_duplicates[user_movie_duplicates["count"] > 1]

print("Number of userId-movieId pairs with more than one rating:",
      repeated_user_movie.shape[0])

display(repeated_user_movie.head(20))


Number of userId-movieId pairs with more than one rating: 0


,userId,movieId,count


*Conclusions*  
No duplicates found in ratings. In training model - drop_duplicates, for pipeline reproducibility reasons only. 

## Tags 

### Tags per user and per user-movie pair

I examined:

- how many tags each user has created in total
- how many tags the same user assigns to the same movie

This is relevant for Assignment 1 because I am now moving toward a user-based collaborative approach.  
In this setting, tags are primarily useful as part of user interaction behavior rather than as the main movie representation.  
This analysis helps determine whether users contribute enough tag information to support user profiles, clustering, and recommendation.


**Tags per user** 

In [ ]:
tags_per_user = (
    tags.groupby("userId")
    .size()
    .reset_index(name="tag_count")
)

display(tags_per_user["tag_count"].describe())
display(tags_per_user.sort_values("tag_count", ascending=False).head(10))


count     25280.000000
mean         92.101068
std        4573.045245
min           1.000000
25%           2.000000
50%           4.000000
75%          18.000000
max      723473.000000
Name: tag_count, dtype: float64

,userId,tag_count
16450,215490,723473
17538,229729,20317
4935,63350,19345
9706,126357,17990
10532,137062,16739
13999,183281,15467
20887,274127,15431
22775,298708,13838
17922,234824,13700
7368,95828,12681


**Tags per user-movie pair**

In [8]:
tags_per_user_movie = (
    tags.groupby(["userId", "movieId"])
    .size()
    .reset_index(name="tag_count")
)

display(tags_per_user_movie["tag_count"].describe())
display(tags_per_user_movie.sort_values("tag_count", ascending=False).head(20))


count    446211.000000
mean          5.217969
std          17.439776
min           1.000000
25%           1.000000
50%           2.000000
75%           5.000000
max         813.000000
Name: tag_count, dtype: float64

,userId,movieId,tag_count
263130,215490,7387,813
276920,215490,122914,796
276922,215490,122918,790
282504,215490,143355,781
296820,215490,268642,763
262607,215490,6803,746
276916,215490,122906,720
291694,215490,176801,713
292229,215490,178827,706
294676,215490,187593,696


userId 215490 seems to have a high nubmer of tags per individual movieId. 

In [47]:
tags[
    (tags["userId"] == 215490) &
    (tags["movieId"] == 7387)
]


,userId,movieId,tag,timestamp
1298850,215490,7387,1970s,1612469473
1298851,215490,7387,1973 harley davidson 125 rcx motorcycle,1612469473
1298852,215490,7387,1977 harley davidson 1200 super glide motorycle,1612469473
1298853,215490,7387,1977 harley davidson fx super glide motorcycle,1612469473
1298854,215490,7387,abandoned shopping mall,1612469469
...,...,...,...,...
1299658,215490,7387,zombie survival,1612469473
1299659,215490,7387,zombie violence,1612469473
1299660,215490,7387,zombie with a gun,1612469472
1299661,215490,7387,zombie with a rifle,1612469472


***conclution Tags per user-movie pair***   

This shows that a small number of users assign extremely many tags, sometimes even to the same movie.  
This may create outliers in a user-based model and could affect clustering.  
For that reason, tag-based user features may need normalization, transformation, or filtering before model training.


**Tags per movie**

In [11]:
tags_per_movie = (
    tags.groupby("movieId")
    .size()
    .reset_index(name="tag_count")
)

display(tags_per_movie["tag_count"].describe())

count    53452.000000
mean        43.558988
std        197.466261
min          1.000000
25%          2.000000
50%          6.000000
75%         18.000000
max      11152.000000
Name: tag_count, dtype: float64

**checking tag count for all movies (including those not in tags-file)**

In [16]:
tags_per_movie_all = (
    movies[["movieId", "title"]]
    .merge(tags_per_movie, on="movieId", how="left")
)

tags_per_movie_all["tag_count"] = tags_per_movie_all["tag_count"].fillna(0).astype(int)

display(tags_per_movie_all["tag_count"].describe())
print("Number of movies with 0 tags:", (tags_per_movie_all["tag_count"] == 0).sum())


count    86537.000000
mean        26.905428
std        156.630058
min          0.000000
25%          0.000000
50%          2.000000
75%          8.000000
max      11152.000000
Name: tag_count, dtype: float64

Number of movies with 0 tags: 33085


The tag distribution per movie is sparse.

- 25% of movies have 0 tags
- the median movie has only 2 tags
- 75% of movies have at most 8 tags

This is still useful background information for Assignment 1, even though the current model direction is user-based rather than movie-based.

The results suggest that movie-level tag information is often weak or missing.  
This matters because sparse movie metadata can limit how much supporting item information is available when building user interaction features.

For that reason, movie-level tag sparsity should be treated as a data limitation, not as the main filtering rule for the current model.


In [21]:
ratings_per_movie = (
    ratings.groupby("movieId")
    .size()
    .reset_index(name="rating_count")
)

tags_per_movie = (
    tags.groupby("movieId")
    .size()
    .reset_index(name="tag_count")
)

movie_signal = (
    movies[["movieId", "title"]]
    .merge(ratings_per_movie, on="movieId", how="left")
    .merge(tags_per_movie, on="movieId", how="left")
)

movie_signal["rating_count"] = movie_signal["rating_count"].fillna(0).astype(int)
movie_signal["tag_count"] = movie_signal["tag_count"].fillna(0).astype(int)

display(movie_signal.head())


,movieId,title,rating_count,tag_count
0,1,Toy Story (1995),76813,1440
1,2,Jumanji (1995),30209,653
2,3,Grumpier Old Men (1995),15820,36
3,4,Waiting to Exhale (1995),3028,13
4,5,Father of the Bride Part II (1995),15801,68


In [22]:
movie_signal[["rating_count", "tag_count"]].corr()


,rating_count,tag_count
rating_count,1.000000,0.770133
tag_count,0.770133,1.000000


**This shows that the number of ratings and the number of tags are correlated**, but it does not show whether high ratings and high tag counts are related.  
For the current user-based direction, this is mainly a background observation about item popularity and signal strength.

In [10]:
tags_per_movie_top = (
    tags_per_movie
    .merge(movies[["movieId", "title"]], on="movieId", how="left")
    .sort_values("tag_count", ascending=False)
)

display(tags_per_movie_top.head(20))


,movieId,tag_count,title
248,260,11152,Star Wars: Episode IV - A New Hope (1977)
14323,79132,8978,Inception (2010)
281,296,8753,Pulp Fiction (1994)
2350,2571,7593,"Matrix, The (1999)"
2723,2959,7459,Fight Club (1999)
19894,109487,7331,Interstellar (2014)
302,318,6839,"Shawshank Redemption, The (1994)"
339,356,4880,Forrest Gump (1994)
6940,7361,4680,Eternal Sunshine of the Spotless Mind (2004)
3931,4226,4675,Memento (2000)


### Filtering decision for the first user-based model

For my current Assignment 1 approach, the model should be based primarily on user interactions rather than movie profiles.

The EDA shows that:
- user activity is highly uneven
- many users have relatively few ratings
- tagging activity is also very uneven across users
- movie-level tag information is often sparse

This suggests that the first filtering decisions should focus mainly on weak user profiles rather than on movie tags alone.

A more reasonable first strategy is:

- filter out users with very few ratings
- optionally consider tag activity as an additional user signal
- keep movie metadata such as genres and tags as supporting information
- only use movie-level cutoffs as secondary safeguards if item signal is extremely weak

A reasonable first threshold to test is therefore:

- users with at least 20 ratings
- and later compare this with a stricter version such as 30 ratings
